In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier


# =========================================================
# 1. 데이터 불러오기
# =========================================================
df = pd.read_csv('dataset/PimaIndiansDiabetes.csv')

print(df.head())
print(df.columns)


# =========================================================
# 2. X, y 분리
# =========================================================
X = df.loc[:, df.columns != 'diabetes']

# diabetes가 pos / neg 문자이므로
# XGBoost에서도 사용할 수 있도록 숫자로 변환
y = df['diabetes'].map({
    'neg': 0,
    'pos': 1
})


# =========================================================
# 3. Train / Test 분리
# =========================================================
train_X, test_X, train_y, test_y = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=1234,
    stratify=y
)


# =========================================================
# 결과 저장
# =========================================================
results = []


# =========================================================
# 4. Decision Tree
# =========================================================
dt = DecisionTreeClassifier(random_state=1234)

dt_param = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 4, 5, 6, 7, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt_grid = GridSearchCV(
    estimator=dt,
    param_grid=dt_param,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

dt_grid.fit(train_X, train_y)

dt_best = dt_grid.best_estimator_

dt_train_acc = dt_best.score(train_X, train_y)
dt_test_acc = dt_best.score(test_X, test_y)

results.append(
    ('Decision Tree', dt_train_acc, dt_test_acc)
)

# 필요하면 확인
# print(dt_grid.best_params_)


# =========================================================
# 5. SVM
# =========================================================
# SVM은 변수들의 단위 차이에 영향을 많이 받으므로
# StandardScaler를 먼저 적용
svm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC())
])

svm_param = [
    {
        'svm__kernel': ['rbf'],
        'svm__C': [0.1, 1, 10, 100],
        'svm__gamma': ['scale', 0.001, 0.01, 0.1, 1]
    },
    {
        'svm__kernel': ['linear'],
        'svm__C': [0.01, 0.1, 1, 10]
    }
]

svm_grid = GridSearchCV(
    estimator=svm_model,
    param_grid=svm_param,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

svm_grid.fit(train_X, train_y)

svm_best = svm_grid.best_estimator_

svm_train_acc = svm_best.score(train_X, train_y)
svm_test_acc = svm_best.score(test_X, test_y)

results.append(
    ('SVM', svm_train_acc, svm_test_acc)
)

# 필요하면 확인
# print(svm_grid.best_params_)


# =========================================================
# 6. Random Forest
# =========================================================
rf = RandomForestClassifier(random_state=1234)

rf_param = {
    'n_estimators': [100, 300, 500],
    'max_depth': [5, 10, 20, None],
    'max_features': ['sqrt', 2, 3, 5],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_param,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

rf_grid.fit(train_X, train_y)

rf_best = rf_grid.best_estimator_

rf_train_acc = rf_best.score(train_X, train_y)
rf_test_acc = rf_best.score(test_X, test_y)

results.append(
    ('Random Forest', rf_train_acc, rf_test_acc)
)

# 필요하면 확인
# print(rf_grid.best_params_)


# =========================================================
# 7. XGBoost
# =========================================================
xgb = XGBClassifier(
    random_state=1234,
    eval_metric='logloss',
    n_jobs=-1
)

xgb_param = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    estimator=xgb,
    param_grid=xgb_param,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

xgb_grid.fit(train_X, train_y)

xgb_best = xgb_grid.best_estimator_

xgb_train_acc = xgb_best.score(train_X, train_y)
xgb_test_acc = xgb_best.score(test_X, test_y)

results.append(
    ('XGBoost', xgb_train_acc, xgb_test_acc)
)

# 필요하면 확인
# print(xgb_grid.best_params_)


# =========================================================
# 8. 과제 제출용 출력
# =========================================================
print()

for name, train_acc, test_acc in results:
    print(f'[Algorithm: {name}]')
    print(f'Train accuracy : {train_acc:.4f}')
    print(f'Test accuracy : {test_acc:.4f}')
    print()

   pregnant  glucose  pressure  triceps  insulin  mass  pedigree  age diabetes
0         6      148        72       35        0  33.6     0.627   50      pos
1         1       85        66       29        0  26.6     0.351   31      neg
2         8      183        64        0        0  23.3     0.672   32      pos
3         1       89        66       23       94  28.1     0.167   21      neg
4         0      137        40       35      168  43.1     2.288   33      pos
Index(['pregnant', 'glucose', 'pressure', 'triceps', 'insulin', 'mass',
       'pedigree', 'age', 'diabetes'],
      dtype='object')

[Algorithm: Decision Tree]
Train accuracy : 0.7691
Test accuracy : 0.7835

[Algorithm: SVM]
Train accuracy : 0.7672
Test accuracy : 0.7922

[Algorithm: Random Forest]
Train accuracy : 0.9832
Test accuracy : 0.7835

[Algorithm: XGBoost]
Train accuracy : 0.8622
Test accuracy : 0.7143



In [3]:
# =========================================================
# 실습과제 2
# Accuracy / Sensitivity / Specificity / Precision / F1
# =========================================================
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)

models = {
    'Decision Tree': dt_best,
    'SVM': svm_best,
    'Random Forest': rf_best,
    'XGBoost': xgb_best
}

metric_results = []


for name, model in models.items():

    # Test 데이터 예측
    pred_y = model.predict(test_X)

    # Accuracy
    accuracy = accuracy_score(test_y, pred_y)

    # Sensitivity = Recall
    sensitivity = recall_score(test_y, pred_y)

    # Precision
    precision = precision_score(test_y, pred_y)

    # F1 score
    f1 = f1_score(test_y, pred_y)

    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(test_y, pred_y).ravel()

    # Specificity
    specificity = tn / (tn + fp)

    metric_results.append([
        name,
        accuracy,
        sensitivity,
        specificity,
        precision,
        f1
    ])


# =========================================================
# 비교 결과 표 생성
# =========================================================

result_df = pd.DataFrame(
    metric_results,
    columns=[
        'Algorithm',
        'Accuracy',
        'Sensitivity (Recall)',
        'Specificity',
        'Precision',
        'F1 Score'
    ]
)

# 소수점 4자리로 표시
result_df = result_df.round(4)

print()

print(result_df.to_string(index=False))


    Algorithm  Accuracy  Sensitivity (Recall)  Specificity  Precision  F1 Score
Decision Tree    0.7835                0.5926       0.8867     0.7385    0.6575
          SVM    0.7922                0.6296       0.8800     0.7391    0.6800
Random Forest    0.7835                0.6667       0.8467     0.7013    0.6835
      XGBoost    0.7143                0.5679       0.7933     0.5974    0.5823
